# अध्याय 7: चैट एप्लिकेशन बनाना  
## OpenAI API त्वरित शुरुआत  

यह नोटबुक [Azure OpenAI सैंपल्स रिपॉजिटरी](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) से अनुकूलित की गई है जिसमें नोटबुक शामिल हैं जो [Azure OpenAI](notebook-azure-openai.ipynb) सेवाओं तक पहुँचती हैं।  

Python OpenAI API कुछ संशोधनों के साथ Azure OpenAI मॉडल के साथ भी काम करता है। यहाँ भेदों के बारे में और जानें: [Python के साथ OpenAI और Azure OpenAI एंडपॉइंट्स के बीच कैसे स्विच करें](https://learn.microsoft.com/azure/ai-services/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)  


# अवलोकन  
"बड़े भाषा मॉडल ऐसे फंक्शन होते हैं जो टेक्स्ट को टेक्स्ट में मैप करते हैं। एक इनपुट टेक्स्ट स्ट्रिंग दिए जाने पर, एक बड़ा भाषा मॉडल अगले आने वाले टेक्स्ट की भविष्यवाणी करने की कोशिश करता है"(1)। यह "क्विकस्टार्ट" नोटबुक उपयोगकर्ताओं को उच्च स्तरीय LLM अवधारणाओं, AML के साथ शुरुआत करने के लिए आवश्यक मुख्य पैकेज आवश्यकताओं, प्रांप्ट डिज़ाइन का एक सरल परिचय, और विभिन्न उपयोग मामलों के कई छोटे उदाहरणों से परिचित कराएगा। 


## सामग्री सूची  

[अवलोकन](#overview)  
[OpenAI सेवा का उपयोग कैसे करें](#how-to-use-openai-service)  
[1. अपनी OpenAI सेवा बनाना](#1.-creating-your-openai-service)  
[2. स्थापना](#2.-installation)    
[3. प्रमाण-पत्र](#3.-credentials)  

[उपयोग के मामले](#use-cases)    
[1. पाठ सारांशित करें](#1.-summarize-text)  
[2. पाठ वर्गीकृत करें](#2.-classify-text)  
[3. नए उत्पाद नाम बनाएं](#3.-generate-new-product-names)  
[4. वर्गीकर्ता को फाइन ट्यून करें](#4.fine-tune-a-classifier)  

[संदर्भ](#references)


### अपना पहला प्रॉम्प्ट बनाएं  
यह छोटा अभ्यास एक सरल कार्य "सारांशण" के लिए OpenAI मॉडल को प्रॉम्प्ट सबमिट करने का मूल परिचय प्रदान करेगा।  


**कदम**:  
1. अपने पायथन वातावरण में OpenAI लाइब्रेरी स्थापित करें  
2. मानक सहायक लाइब्रेरी लोड करें और आपने जो OpenAI सेवा बनाई है उसके लिए आपकी सामान्य OpenAI सुरक्षा क्रेडेंशियल सेट करें  
3. अपने कार्य के लिए एक मॉडल चुनें  
4. मॉडल के लिए एक सरल प्रॉम्प्ट बनाएं  
5. अपने अनुरोध को मॉडल API पर सबमिट करें!  


### 1. OpenAI इंस्टॉल करें


In [ ]:
%pip install openai python-dotenv

### 2. सहायक पुस्तकालयों को आयात करें और प्रमाणपत्रों को इंस्टेंटिएट करें


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. सही मॉडल खोजना  
GPT-4o और GPT-4o मिनी जैसे मॉडल प्राकृतिक भाषा को समझ सकते हैं और उसका उत्पादन कर सकते हैं, और OpenAI प्लेटफ़ॉर्म पर विभिन्न ताकत और गति के साथ उपलब्ध हैं जो अलग-अलग कार्यों के लिए उपयुक्त हैं। 


In [ ]:
# Select a general purpose chat model
model = "gpt-4o-mini"


## 4. प्रॉम्प्ट डिजाइन  

"बड़े भाषा मॉडल का जादू यह है कि विशाल मात्रा में टेक्स्ट पर इस पूर्वानुमान त्रुटि को कम करने के लिए प्रशिक्षित होने के कारण, मॉडल उन पूर्वानुमानों के लिए उपयोगी अवधारणाओं को सीख जाते हैं। उदाहरण के लिए, वे ऐसे अवधारणाएँ सीखते हैं "(1):

* कैसे वर्तनी करें
* व्याकरण कैसे काम करता है
* पैराफ्रेज़ कैसे करें
* प्रश्नों का उत्तर कैसे दें
* बातचीत कैसे करें
* कई भाषाओं में कैसे लिखें
* कोड कैसे करें
* आदि।

#### बड़े भाषा मॉडल को कैसे नियंत्रित करें  
"बड़े भाषा मॉडल में सभी इनपुट्स में से, सबसे प्रभावशाली टेक्स्ट प्रॉम्प्ट है(1)।

बड़े भाषा मॉडल को कुछ तरीकों से आउटपुट उत्पन्न करने के लिए प्रॉम्प्ट किया जा सकता है:

निर्देश: मॉडल को बताएं कि आप क्या चाहते हैं
पूर्णता: मॉडल को उस चीज़ की शुरुआत पूरी करने के लिए प्रेरित करें जो आप चाहते हैं
प्रदर्शन: मॉडल को दिखाएं कि आप क्या चाहते हैं, जिसमें से:
प्रॉम्प्ट में कुछ उदाहरण
ठीक-ठाक प्रशिक्षण डेटासेट में कई सैकड़ों या हजारों उदाहरण"



#### प्रॉम्प्ट बनाने के तीन मूल सिद्धांत हैं:

**दिखाएं और बताएं**। स्पष्ट करें कि आप क्या चाहते हैं, चाहे वह निर्देशों के माध्यम से हो, उदाहरणों के जरिए, या दोनों का संयोजन हो। यदि आप चाहते हैं कि मॉडल आइटमों की सूची को वर्णमाला क्रम में रैंक करे या पैराग्राफ को भावना के अनुसार वर्गीकृत करे, तो इसे दिखाएं कि आप वही चाहते हैं।

**गुणवत्ता वाले डेटा प्रदान करें**। यदि आप कोई वर्गीकर्ता बनाना चाहते हैं या मॉडल को किसी पैटर्न का पालन करना चाहते हैं, तो सुनिश्चित करें कि पर्याप्त उदाहरण हों। अपने उदाहरणों को ध्यानपूर्वक प्रूफरीड करें—मॉडल आमतौर पर बेसिक वर्तनी की गलतियों को समझ सकता है और आपको प्रतिक्रिया दे सकता है, लेकिन यह यह भी मान सकता है कि यह जानबूझकर किया गया है और इसका प्रभाव प्रतिक्रिया पर पड़ सकता है।

**अपनी सेटिंग्स जांचें।** तापमान और top_p सेटिंग्स नियंत्रित करती हैं कि मॉडल प्रतिक्रिया उत्पन्न करने में कितना निर्धारित है। यदि आप मॉडल से ऐसा उत्तर पूछ रहे हैं जिसके लिए केवल एक सही जवाब है, तो आप इन्हें कम सेट करना चाहेंगे। यदि आप अधिक विविध प्रतिक्रियाएँ चाहते हैं, तो आप इन्हें अधिक सेट करना चाहेंगे। लोग इन सेटिंग्स के साथ जो सबसे बड़ी गलती करते हैं वह यह मान लेना कि ये "चतुराई" या "रचनात्मकता" के नियंत्रण हैं।


स्रोत: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. सबमिट करें!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### एक ही कॉल को दोहराएं, परिणाम कैसे तुलना करते हैं?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## टेक्स्ट संक्षेप करें  
#### चुनौती  
एक टेक्स्ट अनुच्छेद के अंत में 'tl;dr:' जोड़कर टेक्स्ट का संक्षेप करें। ध्यान दें कि मॉडल बिना किसी अतिरिक्त निर्देश के कई कार्यों को कैसे समझता है और करता है। आप मॉडल के व्यवहार को संशोधित करने और प्राप्त संक्षेप को अनुकूलित करने के लिए tl;dr से अधिक वर्णनात्मक संकेतों के साथ प्रयोग कर सकते हैं(3)।  

हाल के कार्यों ने यह साबित किया है कि बड़ी कॉर्पस पर प्री-ट्रेनिंग करके और इसके बाद किसी विशिष्ट कार्य पर फाइन-ट्यूनिंग करके कई NLP कार्यों और बेंचमार्क में महत्वपूर्ण सुधार होते हैं। जबकि वास्तुकला में आमतौर पर कार्य-निरपेक्ष होता है, इस विधि के लिए हजारों या दसियों हजार उदाहरणों वाले कार्य-विशिष्ट फाइन-ट्यूनिंग डेटासेट की आवश्यकता होती है। इसके विपरीत, मनुष्य आमतौर पर केवल कुछ उदाहरणों या सरल निर्देशों से नए भाषा कार्य को कर सकते हैं - ऐसा कुछ जो वर्तमान NLP सिस्टम अभी भी काफी हद तक करने में संघर्ष कर रहे हैं। यहां हम दिखाते हैं कि भाषा मॉडल का आकार बढ़ाने से कार्य-निरपेक्ष, कुछ-शॉट प्रदर्शन में भारी सुधार होता है, कभी-कभी पूर्व राज्य-के-कला फाइन-ट्यूनिंग विधियों के साथ प्रतिस्पर्धा तक पहुंच जाता है। 



सारांश  


# कई उपयोग मामलों के लिए अभ्यास  
1. पाठ सारांशित करें  
2. पाठ वर्गीकृत करें  
3. नए उत्पाद नाम उत्पन्न करें


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## पाठ वर्गीकृत करें  
#### चुनौती  
आइटमों को दिए गए श्रेणियों में वर्गीकृत करें जो अनुमान लगाने के समय प्रदान की जाती हैं। निम्न उदाहरण में, हम श्रेणियाँ और वर्गीकृत करने के लिए पाठ दोनों ही प्रॉम्प्ट (*playground_reference) में प्रदान करते हैं। 

ग्राहक पूछताछ: नमस्ते, मेरी लैपटॉप कीबोर्ड की एक कुंजी हाल ही में टूट गई है और मुझे उसका प्रतिस्थापन चाहिए:  

वर्गीकृत श्रेणी:  


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## नए उत्पाद नाम बनाएँ
#### चुनौती
उदाहरण शब्दों से उत्पाद नाम बनाएँ। यहाँ हम प्रॉम्प्ट में उस उत्पाद के बारे में जानकारी शामिल करते हैं जिसके लिए हम नाम उत्पन्न करने जा रहे हैं। हम एक समान उदाहरण भी प्रदान करते हैं ताकि वह पैटर्न दिखा सकें जो हमें प्राप्त करना है। हमने अधिक यादृच्छिकता और अधिक नवीन प्रतिक्रियाओं के लिए तापमान मान भी उच्च सेट किया है।

उत्पाद विवरण: एक घरेलू मिल्कशेक मेकर
बीज शब्द: तेज, स्वस्थ, कॉम्पैक्ट।
उत्पाद नाम: HomeShaker, Fit Shaker, QuickShake, Shake Maker

उत्पाद विवरण: एक जोड़ी जूते जो किसी भी पैर के आकार के लिए अनुकूल हो सकते हैं।
बीज शब्द: अनुकूलनीय, फिट, ऑम्नी-फिट।


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# संदर्भ  
- [Openai कुकबुक](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry पोर्टल](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [टेक्स्ट वर्गीकृत करने के लिए GPT मॉडलों को फाइन-ट्यून करने के सर्वोत्तम अभ्यास](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# और अधिक सहायता के लिए  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# योगदानकर्ता
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
